In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
import re

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.chronos_bolt.circuitlens import CircuitLensBolt
from tsfm_lens.timesfm.circuitlens import CircuitLensTimesFM
from tsfm_lens.toto.circuitlens import CircuitLensToto
from tsfm_lens.utils import (
    apply_custom_style,
    diagnose_attention,
    extract_projection_weights_Chronos,
    extract_projection_weights_TimesFM2p5,
    extract_projection_weights_Toto,
    plot_combined_corner,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

### Load Pipeline

In [ ]:
model_name = "Chronos"

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "head_outputs", model_name)
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
save_figs = True

In [ ]:
if model_name == "Chronos":
    pipeline = CircuitLensChronos.from_pretrained(
        "amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16
    )
    model = pipeline.model.model
elif model_name == "Chronos Bolt":
    pipeline = CircuitLensBolt.from_pretrained(
        "amazon/chronos-bolt-base", device_map=device, torch_dtype=torch.bfloat16
    )
    model = pipeline.model
elif model_name == "Toto":
    pipeline = CircuitLensToto("Datadog/Toto-Open-Base-1.0", device_map=device)
    model = pipeline.model
elif model_name == "TimesFM 2.5":
    pipeline = CircuitLensTimesFM("google/timesfm-2.5-200m-pytorch", device_map=device)
    model = pipeline.model.model
else:
    raise ValueError(f"Model name {model_name} not supported")

num_layers: int = pipeline.num_layers
num_heads: int = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

### Utils

### Analyze Singular Values and More

In [ ]:
# selected_layers = [0]
selected_layers = range(num_layers)
selected_component_chronos = "CA"  # either SA for self-attention or CA for cross-attention

In [ ]:
if model_name in ["Chronos", "Chronos Bolt"]:
    all_heads = [
        (layer_idx, head_idx, selected_component_chronos)
        for layer_idx in selected_layers
        for head_idx in range(num_heads)
    ]
else:
    all_heads = [(layer_idx, head_idx) for layer_idx in selected_layers for head_idx in range(num_heads)]

In [ ]:
all_heads

In [ ]:
reports_fname = f"{pipeline.__class__.__name__}_reports.pkl"

if os.path.exists(reports_fname):
    print(f"Loading reports from {reports_fname}")
    with open(reports_fname, "rb") as f:
        reports = pickle.load(f)
else:
    # First, collect all reports
    reports = {}
    for head_to_analyze in all_heads:
        print("=" * 100)
        if model_name in ["Chronos", "Chronos Bolt"]:
            selected_layer, selected_head, selected_component = head_to_analyze  # type: ignore
            print(f"Analyzing Layer {selected_layer} Head {selected_head} and component {selected_component}")
            head_WQ, head_WK, _, _ = extract_projection_weights_Chronos(
                model,  # type: ignore
                selected_layer=selected_layer,
                selected_component=selected_component,  # type: ignore
                selected_head=selected_head,
                verbose=False,
            )
            key = f"L{selected_layer}_H{selected_head}_{selected_component}"
        else:
            selected_layer, selected_head = head_to_analyze  # type: ignore
            print(f"Analyzing Layer {selected_layer} Head {selected_head}")
            if model_name == "Toto":
                head_WQ, head_WK, _, _ = extract_projection_weights_Toto(
                    model,  # type: ignore
                    selected_layer=selected_layer,
                    selected_head=selected_head,
                    verbose=False,
                )
            elif model_name == "TimesFM 2.5":
                head_WQ, head_WK, _, _ = extract_projection_weights_TimesFM2p5(
                    model,  # type: ignore
                    selected_layer=selected_layer,
                    selected_head=selected_head,
                    verbose=False,
                )
            else:
                raise ValueError(f"Model name {model_name} not supported")
            key = f"L{selected_layer}_H{selected_head}"

        # W_Q, W_K have shape (d_k, d_h). H has shape (T, d_h) if you want softmax diagnostics.
        report = diagnose_attention(head_WQ, head_WK, H=None, scale_by_sqrt_dk=True, top_k=20)
        print(report["human_readable_summary"])
        reports[key] = report

    # save reports
    with open(f"{pipeline.__class__.__name__}_reports.pkl", "wb") as f:
        pickle.dump(reports, f)


In [ ]:
srank_fname = f"{pipeline.__class__.__name__}_stable_rank.pkl"

if os.path.exists(srank_fname):
    print(f"Loading stable rank from {srank_fname}")
    with open(srank_fname, "rb") as f:
        stable_rank = pickle.load(f)
else:
    # compute the stable rank of all the heads
    stable_rank = {}
    for key, report in reports.items():
        s = report["singular_values"]
        # stable rank is froebenius norm / spectral norm
        # so it is sum of squared singular values / square of largest singular value
        stable_rank[key] = np.sum(s**2) / s[0] ** 2

    # save stable_rank
    with open(f"{pipeline.__class__.__name__}_stable_rank.pkl", "wb") as f:
        pickle.dump(stable_rank, f)


In [ ]:
# save a list to a file of all keys in stable_rank sorted from lowest to highest stable rank
stable_rank_keys = sorted(stable_rank.keys(), key=lambda x: stable_rank[x])
with open(f"{pipeline.__class__.__name__}_srank_low_to_high.txt", "w") as f:
    for key in stable_rank_keys:
        f.write(f"{key}: {stable_rank[key]:.4f}\n")


In [ ]:
reports_by_layer = {
    layer: {key: report for key, report in reports.items() if f"L{layer}" in key} for layer in selected_layers
}

In [ ]:
# save stable rank keys per layer, sorted from lowest to highest stable rank
import json

srank_by_layer_fname = f"{pipeline.__class__.__name__}_srank_low_to_high_by_layer.json"

# if os.path.exists(srank_by_layer_fname):
#     print(f"Loading srank by layer from {srank_by_layer_fname}")
#     with open(srank_by_layer_fname) as f:
#         all_layers_data = json.load(f)
# else:
all_layers_data = {}
for layer in selected_layers:
    stable_rank_keys_layer = sorted(reports_by_layer[layer].keys(), key=lambda x: stable_rank[x])

    layer_data = {int(key.split("_")[1][1:]): round(stable_rank[key], 4) for key in stable_rank_keys_layer}
    all_layers_data[int(layer)] = layer_data

with open(f"{pipeline.__class__.__name__}_srank_low_to_high_by_layer.json", "w") as f:
    json.dump(all_layers_data, f, indent=2)


In [ ]:
layer_to_visualize = None

In [ ]:
if layer_to_visualize is not None:
    stable_ranks_list = [stable_rank[key] for key in reports_by_layer[layer_to_visualize].keys()]
else:
    stable_ranks_list = [stable_rank[key] for key in reports.keys()]


In [ ]:
k_top, k_bottom = 18, 18

In [ ]:
# Plot 1: Singular values spectrum
fig, ax = plt.subplots(figsize=(6, 5))

# Select top k and bottom k keys by stable rank
relevant_stable_rank = (
    {key: stable_rank[key] for key in reports_by_layer[layer_to_visualize].keys()}
    if layer_to_visualize is not None
    else stable_rank
)
sorted_keys = sorted(relevant_stable_rank.keys(), key=lambda x: relevant_stable_rank[x], reverse=True)
combined_keys = sorted(
    set(sorted_keys[:k_top] + sorted_keys[-k_bottom:]), key=lambda x: relevant_stable_rank[x], reverse=True
)

# Create a colormap based on stable rank
stable_rank_values = list(relevant_stable_rank.values())
norm = plt.Normalize(vmin=min(stable_rank_values), vmax=max(stable_rank_values))  # type: ignore
cmap = getattr(plt.cm, "viridis")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

for key in combined_keys:
    report = reports[key]
    s = report["singular_values"]
    k = min(30, len(s))
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    color = cmap(norm(stable_rank[key]))
    ax.plot(np.arange(1, k + 1), s[:k], marker="o", color=color, label=pretty_key, alpha=0.7)

ax.set_xlabel("Index r", fontweight="bold")
ax.set_ylabel(r"$\sigma_r$", fontweight="bold")
title = r"Singular Values of $W_Q^\top W_K$ per Head"
if layer_to_visualize is not None:
    title += f" (Layer {layer_to_visualize})"
ax.set_title(title, fontweight="bold")
plt.colorbar(sm, ax=ax, label="Stable Rank", shrink=0.8)
plt.legend(frameon=True, loc="upper right", fontsize=6.5, ncol=4)
ax.grid(True, alpha=0.3)
# log scale y axis
# ax.set_yscale("log")
plt.tight_layout()
if save_figs:
    plt.savefig(
        os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_singular_values_spectrum.pdf"), bbox_inches="tight"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Select top k and bottom k keys by stable rank
relevant_stable_rank = (
    {key: stable_rank[key] for key in reports_by_layer[layer_to_visualize].keys()}
    if layer_to_visualize is not None
    else stable_rank
)
sorted_keys = sorted(relevant_stable_rank.keys(), key=lambda x: relevant_stable_rank[x], reverse=True)
keys_top_stable_rank = sorted_keys[:k_top]
keys_bottom_stable_rank = sorted_keys[-k_bottom:]
combined_keys = sorted(
    set(keys_top_stable_rank + keys_bottom_stable_rank), key=lambda x: relevant_stable_rank[x], reverse=True
)

# Create a colormap based on stable rank
stable_rank_values = list(relevant_stable_rank.values())
norm = plt.Normalize(vmin=min(stable_rank_values), vmax=max(stable_rank_values))  # type: ignore
cmap = getattr(plt.cm, "viridis")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

for key in combined_keys:
    explained_energy = reports[key]["explained_energy"]
    indices_considered = np.arange(1, len(explained_energy) + 1)
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    color = cmap(norm(stable_rank[key]))
    ax.plot(indices_considered, explained_energy, marker="o", color=color, label=pretty_key, alpha=0.7)

ax.set_xlabel("Top singular values", fontweight="bold")
ax.set_xticks(indices_considered)
title = "Explained Energy (Frobenius Norm)"
if layer_to_visualize is not None:
    title += f" (Layer {layer_to_visualize})"
ax.set_title(title, fontweight="bold")
plt.colorbar(sm, ax=ax, label="Stable Rank", shrink=0.8)
plt.legend(frameon=True, loc="lower right", fontsize=6.5, ncol=4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
if save_figs:
    plt.savefig(os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_explained_energy.pdf"), bbox_inches="tight")

In [ ]:
# Convert keys from "L{layer}_H{head}_CA" format to (layer, head) tuples
highest_stable_rank_heads = [(int(m[0]), int(m[1])) for key in keys_top_stable_rank if (m := re.findall(r"\d+", key))]
print("(layer, head) tuples for top stable rank (decreasing order) \n", highest_stable_rank_heads)

In [ ]:
lowest_stable_rank_heads = [
    (int(m[0]), int(m[1])) for key in reversed(keys_bottom_stable_rank) if (m := re.findall(r"\d+", key))
]
print("(layer, head) tuples for bottom stable rank (increasing order) \n", lowest_stable_rank_heads)

In [ ]:
reports.keys()

In [ ]:
selected_head = keys_top_stable_rank[0]

selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)
print(f"pretty_key: {pretty_key}")
# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key} with stable rank {stable_rank[selected_head]:.2f}: Singular Subspaces: U (Output Space) and V (Input Space)",
    xlim=(-0.2, 0.2),
    ylim=(-0.2, 0.2),
    save_path=os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_{pretty_key}_singular_subspaces.pdf")
    if save_figs
    else None,
)
print(f"pretty_key: {pretty_key}")

In [ ]:
selected_head = keys_bottom_stable_rank[-1]
selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key} with stable rank {stable_rank[selected_head]:.2f}: Singular Subspaces: U (Output Space) and V (Input Space)",
    xlim=(-0.2, 0.2),
    ylim=(-0.2, 0.2),
    save_path=os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_{pretty_key}_singular_subspaces.pdf")
    if save_figs
    else None,
)
print(f"pretty_key: {pretty_key}")


In [ ]:
srank_fname = f"{pipeline.__class__.__name__}_alignment_scores.pkl"

if os.path.exists(srank_fname):
    print(f"Loading stable rank from {srank_fname}")
    with open(srank_fname, "rb") as f:
        alignment_scores = pickle.load(f)
else:
    # compute the stable rank of all the heads
    alignment_scores = {}
    for key, report in reports.items():
        alignment_scores[key] = np.mean(np.cos(report["principal_angles_radians"]) ** 2)
    # save stable_rank
    with open(f"{pipeline.__class__.__name__}_alignment_scores.pkl", "wb") as f:
        pickle.dump(alignment_scores, f)


In [ ]:
from matplotlib.colors import Normalize

# Choose coloring metric: 'stable_rank' or 'alignment'
color_by = "layer"  # Change to 'alignment' to color by normalized alignment

if color_by == "stable_rank":
    sorted_reports = sorted(reports.items(), key=lambda x: stable_rank[x[0]])
    color_values = np.array([stable_rank[key] for key, _ in sorted_reports])
    colorbar_label = "Stable Rank"
    colors = getattr(plt.cm, "viridis")(np.linspace(0, 1, len(sorted_reports)))

elif color_by == "alignment":
    sorted_reports = sorted(reports.items(), key=lambda x: alignment_scores[x[0]])
    color_values = np.array([alignment_scores[key] for key, _ in sorted_reports])
    colorbar_label = "Normalized Alignment"
    colors = getattr(plt.cm, "viridis")(np.linspace(0, 1, len(sorted_reports)))

elif color_by == "layer":
    # Sort by layer index (L0_H0, L0_H1, ..., L1_H0, L1_H1, ...)
    def extract_layer_head(key):
        match = re.match(r"L(\d+)_H(\d+)", key)
        if match:
            return (int(match.group(1)), int(match.group(2)))
        return (0, 0)

    sorted_reports = sorted(reports.items(), key=lambda x: extract_layer_head(x[0]))
    # For layer-based coloring, assign same color to all heads in a layer
    # Extract layer index for each head
    layer_indices = [extract_layer_head(key)[0] for key, _ in sorted_reports]

    # Assign the same color value (layer index) to all heads in the same layer
    color_values = np.array(layer_indices, dtype=float)
    colorbar_label = "Layer Index"
    layer_colors = getattr(plt.cm, "viridis")(np.linspace(0, 1, num_layers))
    colors = np.repeat(layer_colors, num_heads, axis=0)
else:
    raise ValueError(f"Invalid color_by option: {color_by}")


# Plot 3: Principal angles
fig, ax = plt.subplots(figsize=(6, 5))
for idx, (key, report) in enumerate(sorted_reports):
    angles_rad = report["principal_angles_radians"]
    if angles_rad.size > 0:
        ax.plot(np.degrees(angles_rad), marker="o", color=colors[idx], label=key, alpha=0.5, markersize=0)

ax.set_xlabel("Head Dimension", fontweight="bold")
ax.set_ylabel("Angle (degrees)", fontweight="bold")
ax.set_title(f"Principal Angles For All Heads ({model_name})", fontweight="bold")
ax.grid(True, alpha=0.3)
ax.set_aspect("auto")

# Add colorbar
sm = plt.cm.ScalarMappable(cmap="viridis", norm=Normalize(vmin=color_values.min(), vmax=color_values.max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.8)
cbar.set_label(colorbar_label, fontweight="bold")

# Create two-column legend with highest and lowest values
top_10_highest = sorted_reports[-10:][::-1]  # Highest 10, sorted from highest to lowest
top_10_lowest = sorted_reports[:10]  # Lowest 10, sorted from lowest to highest

# Create legend handles and labels
legend_handles = []
legend_labels = []

# Add highest values
legend_labels.append("Top 10 Heads")
legend_handles.append(plt.Line2D([0], [0], color="none"))
for key, _ in top_10_highest:
    idx = [k for k, _ in sorted_reports].index(key)
    value = color_values[idx]
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key).replace("_", " ")
    legend_labels.append(f"{pretty_key}: {value:.2f}")
    legend_handles.append(plt.Line2D([0], [0], color=colors[idx], linewidth=2))

# Add lowest values
legend_labels.append("Bottom 10 Heads")
legend_handles.append(plt.Line2D([0], [0], color="none"))
for key, _ in top_10_lowest:
    idx = [k for k, _ in sorted_reports].index(key)
    value = color_values[idx]
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key).replace("_", " ")
    legend_labels.append(f"{pretty_key}: {value:.2f}")
    legend_handles.append(plt.Line2D([0], [0], color=colors[idx], linewidth=2))

ax.legend(legend_handles, legend_labels, ncol=2, fontsize=6, loc="lower right", frameon=True)

plt.tight_layout()
if save_figs:
    plt.savefig(
        os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_principal_angles_sorted_by_{color_by}.pdf"),
        bbox_inches="tight",
    )


In [ ]:
stable_rank_keys[-1]

In [ ]:
stable_rank_keys[len(stable_rank_keys) // 2]

In [ ]:
# layers_to_plot = [0, 2, 5, 10, 15, 18, 19]
layers_to_plot = [0, 7, 8, 11]
colors_by_layer = plt.cm.viridis(np.linspace(0, 1, num_layers))
# Create line plot of stable rank for all layers (sorted from smallest to largest)
fig, ax = plt.subplots(figsize=(5, 5))

for i, layer_idx in enumerate(layers_to_plot):
    # Get stable rank for all heads in this layer
    layer_stable_ranks = []
    for head_idx in range(num_heads):
        key = f"L{layer_idx}_H{head_idx}"
        if model_name in ["Chronos", "Chronos Bolt"]:
            key = f"L{layer_idx}_H{head_idx}_CA"
        if key in stable_rank:
            layer_stable_ranks.append(stable_rank[key])

    # Sort stable ranks from smallest to largest
    layer_stable_ranks_sorted = sorted(layer_stable_ranks)

    # Plot line for this layer
    ax.plot(
        range(len(layer_stable_ranks_sorted)),
        layer_stable_ranks_sorted,
        marker="o",
        linewidth=2,
        markersize=4,
        color=colors_by_layer[layer_idx],
        label=f"Layer {layer_idx}",
        alpha=0.7,
    )

ax.set_xlabel("Head Index (sorted)", fontweight="bold")
ax.set_ylabel("Stable Rank", fontweight="bold")
ax.set_title("Stable Rank by Layer (Sorted)", fontweight="bold")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8, loc="best")

plt.tight_layout()
if save_figs:
    plt.savefig(
        os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_stable_rank_lineplot_all_layers.pdf"),
        bbox_inches="tight",
    )


In [ ]:
# Sort by stable rank and calculate normalized alignments
sorted_reports = sorted(reports.items(), key=lambda x: stable_rank[x[0]])
stable_rank_values = np.array([stable_rank[key] for key, _ in sorted_reports])
normalized_alignments = [
    np.sum(np.cos(report["principal_angles_radians"]) ** 2) / len(report["principal_angles_radians"])
    for _, report in sorted_reports
]

# Plot normalized alignment vs stable rank
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(stable_rank_values, normalized_alignments, s=50, alpha=0.7)
ax.set_xlabel("Stable Rank", fontweight="bold")
ax.set_ylabel("Normalized Alignment", fontweight="bold")
ax.set_title(f"Normalized Alignment vs Stable Rank ({model_name})", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
if save_figs:
    plt.savefig(
        os.path.join(plot_save_dir, f"{pipeline.__class__.__name__}_normalized_alignment_vs_stable_rank.pdf"),
        bbox_inches="tight",
    )


In [ ]:
alignment_scores.keys()

In [ ]:
# save a list to a file of all keys in alignment_scores sorted from lowest to highest alignment score
alignment_score_keys = sorted(alignment_scores.keys(), key=lambda x: alignment_scores[x])
with open(f"{pipeline.__class__.__name__}_alignment_low_to_high.txt", "w") as f:
    for key in alignment_score_keys:
        f.write(f"{key}: {alignment_scores[key]:.4f}\n")


In [ ]:
# save alignment score keys per layer, sorted from lowest to highest alignment score
import json

alignment_by_layer_fname = f"{pipeline.__class__.__name__}_alignment_low_to_high_by_layer.json"

if os.path.exists(alignment_by_layer_fname):
    print(f"Loading alignment by layer from {alignment_by_layer_fname}")
    with open(alignment_by_layer_fname) as f:
        all_layers_data = json.load(f)
else:
    all_layers_data = {}
    for layer in selected_layers:
        # Filter keys for this layer
        layer_keys = [key for key in alignment_scores.keys() if key.startswith(f"L{layer}_")]
        alignment_keys_layer = sorted(layer_keys, key=lambda x: alignment_scores[x])

        layer_data = {
            int(key.split("_")[1][1:]): float(round(alignment_scores[key], 4)) for key in alignment_keys_layer
        }
        all_layers_data[int(layer)] = layer_data

    with open(f"{pipeline.__class__.__name__}_alignment_low_to_high_by_layer.json", "w") as f:
        json.dump(all_layers_data, f, indent=2)
